# Healthcare Operations Analysis
### *Does Who Runs Your Hospital Affect How Well It Performs?*

**Dataset**: CMS Hospital General Information (January 2026 release)  
**Source**: Centers for Medicare & Medicaid Services (Care Compare)

---

This notebook investigates whether structural factors such as **ownership type**, **hospital classification**, and **geography**, are associated with measurable differences in care quality across five clinical domains:

| Domain | What It Measures |
|--------|-----------------|
| Mortality | 30-day death rates |
| Safety | Hospital-acquired infections/surgical complications |
| Readmission | Unplanned readmissions within 30 days |

Each domain reports how many measures a hospital scored **Better**, **No Different**, or **Worse** than the national average. From these raw counts we develop custom KPIs to compare geographies, rank hospitals, and classify risk.

### Custom KPIs Developed in This Analysis

| KPI | Definition | Introduced In |
|-----|------------|---------------|
| **Domain Better Rate** | Proportion of measures rated Better within a domain | Section 2 |
| **Net Domain Score** | Better measure count − Worse measure count per domain | Section 3 |
| **Risk Score & Category** | Count of domains rated Worse; flagged At-Risk if ≥ 2 | Section 4 |
| **Composite Quality Score** | Sum of net scores across Mortality, Safety, and Readmission | Section 5 |

---


## Setup & Data Load

In [1]:
import sqlite3
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Load CSV
df = pd.read_csv("Hospital_General_Information.csv")
print(f"Rows: {len(df):,}  |  Columns: {len(df.columns)}")

# Load into SQLite in-memory database
conn = sqlite3.connect(":memory:")
df.to_sql("hospitals", conn, if_exists="replace", index=False)

def q(sql):
    """Execute raw SQL and return a pandas DataFrame."""
    return pd.read_sql_query(sql, conn)

print("SQLite loaded. Ready to query.")


Rows: 5,426  |  Columns: 38
SQLite loaded. Ready to query.


### Dataset Overview

In [2]:
overview = q("""
SELECT
    COUNT(*)                                                            AS total_hospitals,
    SUM(CASE WHEN "Hospital overall rating" NOT IN ('Not Available', '') THEN 1 ELSE 0 END)
                                                                        AS rated_hospitals,
    COUNT(DISTINCT State)                                               AS states,
    COUNT(DISTINCT "Hospital Type")                                     AS hospital_types,
    COUNT(DISTINCT "Hospital Ownership")                                AS ownership_types,
    SUM(CASE WHEN "Emergency Services" = 'Yes' THEN 1 ELSE 0 END)      AS with_emergency,
    SUM(CASE WHEN "Meets criteria for birthing friendly designation" = 'Y' THEN 1 ELSE 0 END)
                                                                        AS birthing_friendly
FROM hospitals
""")
overview.T.rename(columns={0: "Value"})


,Value
total_hospitals,5426
rated_hospitals,2866
states,56
hospital_types,8
ownership_types,12
with_emergency,4499
birthing_friendly,2265


---
## Section 1 — Understanding the data


### Q1 — What is the distribution of overall hospital ratings?


In [3]:
rating_dist = q("""
SELECT
    "Hospital overall rating"           AS rating,
    COUNT(*)                            AS hospital_count,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM hospitals
WHERE "Hospital overall rating" NOT IN ('Not Available', '')
GROUP BY rating
ORDER BY rating
""")

fig = px.bar(
    rating_dist,
    x="rating", y="hospital_count",
    text="pct",
    color="rating",
    color_discrete_sequence=px.colors.sequential.Blues[2:],
    labels={"rating": "Star Rating (1=Lowest, 5=Highest)", "hospital_count": "Number of Hospitals"},
    title="Distribution of CMS Hospital Star Ratings (Rated Hospitals Only)",
)
fig.update_traces(texttemplate="%{text}%", textposition="outside")
fig.update_layout(showlegend=False, xaxis_title="Star Rating", yaxis_title="Hospital Count")
fig.show()
print(rating_dist.to_string(index=False))


rating  hospital_count  pct
     1             229  8.0
     2             649 22.6
     3             935 32.6
     4             765 26.7
     5             288 10.0


### Q2 — Which states have the highest and lowest average hospital ratings?


In [4]:
state_ratings = q("""
SELECT
    State,
    ROUND(AVG(CAST("Hospital overall rating" AS REAL)), 2)  AS avg_rating,
    COUNT(*)                                                AS rated_count,
    SUM(CASE WHEN "Hospital overall rating" = '5' THEN 1 ELSE 0 END) AS five_star,
    SUM(CASE WHEN "Hospital overall rating" = '1' THEN 1 ELSE 0 END) AS one_star
FROM hospitals
WHERE "Hospital overall rating" NOT IN ('Not Available', '')
GROUP BY State
HAVING rated_count >= 5
ORDER BY avg_rating DESC
""")

fig = px.choropleth(
    state_ratings,
    locations="State",
    locationmode="USA-states",
    color="avg_rating",
    scope="usa",
    color_continuous_scale="RdYlGn",
    range_color=[1.5, 3.5],
    labels={"avg_rating": "Avg Star Rating"},
    title="Average Hospital Star Rating by State (min. 5 rated hospitals)",
    hover_data={"rated_count": True, "five_star": True, "one_star": True},
)
fig.update_layout(coloraxis_colorbar=dict(title="Avg Rating"))
fig.show()

print("Top 10 States:")
print(state_ratings.head(10).to_string(index=False))
print("\nBottom 10 States:")
print(state_ratings.tail(10).to_string(index=False))


Top 10 States:
State  avg_rating  rated_count  five_star  one_star
   UT        4.09           23          9         0
   SD        4.08           12          3         0
   CO        3.87           45         12         1
   MN        3.80           41          9         0
   WI        3.77           66         13         1
   MT        3.63           16          3         1
   ID        3.59           17          4         1
   ND        3.56            9          1         0
   VA        3.46           74          7         1
   KS        3.45           42          8         1

Bottom 10 States:
State  avg_rating  rated_count  five_star  one_star
   KY        2.71           56          2         5
   GA        2.65           79          4        11
   WV        2.63           27          2         7
   AR        2.63           40          1         6
   AL        2.57           51          2         5
   NY        2.53          131         12        29
   MS        2.49           45

> **Insight**: Geography matters and likely reflects differences in hospital mix, funding, and patient demographics.


### Q3 — How do ratings vary by hospital type?


In [5]:
type_ratings = q("""
SELECT
    "Hospital Type"                                                         AS hospital_type,
    COUNT(*)                                                                AS total,
    SUM(CASE WHEN "Hospital overall rating" NOT IN ('Not Available', '')
             THEN 1 ELSE 0 END)                                            AS rated_count,
    ROUND(AVG(CASE WHEN "Hospital overall rating" NOT IN ('Not Available', '')
                   THEN CAST("Hospital overall rating" AS REAL) END), 2)   AS avg_rating,
    SUM(CASE WHEN "Hospital overall rating" = '5' THEN 1 ELSE 0 END)       AS five_star,
    SUM(CASE WHEN "Hospital overall rating" = '1' THEN 1 ELSE 0 END)       AS one_star
FROM hospitals
GROUP BY hospital_type
ORDER BY avg_rating DESC NULLS LAST
""")

fig = px.bar(
    type_ratings.dropna(subset=["avg_rating"]),
    x="avg_rating", y="hospital_type",
    orientation="h",
    color="avg_rating",
    color_continuous_scale="RdYlGn",
    range_color=[2, 3.5],
    text="avg_rating",
    labels={"avg_rating": "Avg Star Rating", "hospital_type": "Hospital Type"},
    title="Average Star Rating by Hospital Type",
)
fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig.update_layout(showlegend=False, yaxis=dict(autorange="reversed"))
fig.show()
print(type_ratings.to_string(index=False))


                       hospital_type  total  rated_count  avg_rating  five_star  one_star
Acute Care - Veterans Administration    132          113        4.20         55         0
           Critical Access Hospitals   1376          213        3.12         17        16
                Acute Care Hospitals   3116         2540        3.03        216       213
            Rural Emergency Hospital     39            0         NaN          0         0
                         Psychiatric    633            0         NaN          0         0
                           Long-term      4            0         NaN          0         0
                           Childrens     94            0         NaN          0         0
  Acute Care - Department of Defense     32            0         NaN          0         0


> **Insight**: Average ratings for Acute Care and Critical Access hospitals are close (about 3.0 vs 3.1). The gap is smaller than expected, suggesting some rural Critical Access facilities perform competitively despite tighter resource constraints.


---
## Section 2 — Ownership & Quality: Does Money Change Outcomes?

Hospitals operate under radically different incentive structures. Non-profits must reinvest surplus into care. For-profit hospitals answer to shareholders. Government hospitals serve public mandates. Do these structural differences show up in quality metrics?

We group ownership into four categories:
- **Non-Profit**: Voluntary non-profit (Private, Church, Other)
- **For-Profit**: Proprietary + Physician-owned
- **Government**: Federal, State, Local, Hospital District
- **VA/DoD**: Veterans Administration + Department of Defense

### KPI Development — Domain Performance Rates

This section introduces our first custom KPIs, computed in SQL from CMS Better/Worse measure counts:

- **Domain Better Rate** (`mort_better_rate`, `safety_better_rate`, `readm_better_rate`): proportion of a hospital's measures in each domain rated **Better** than the national average


### Q4 — Do non-profit hospitals outperform proprietary ones across quality domains?


In [6]:
ownership_domains = q("""
SELECT
    CASE
        WHEN "Hospital Ownership" LIKE '%non-profit%'                           THEN 'Non-Profit'
        WHEN "Hospital Ownership" IN ('Proprietary', 'Physician')               THEN 'For-Profit'
        WHEN "Hospital Ownership" LIKE 'Government%' OR "Hospital Ownership" = 'Tribal'
                                                                                THEN 'Government'
        WHEN "Hospital Ownership" LIKE 'Veterans%' OR "Hospital Ownership" = 'Department of Defense'
                                                                                THEN 'VA / DoD'
        ELSE 'Other'
    END AS ownership_group,
    COUNT(*) AS hospital_count,
    ROUND(AVG(
        CAST(NULLIF("Count of MORT Measures Better",  'Not Available') AS REAL) /
        NULLIF(CAST(NULLIF("Count of Facility MORT Measures", 'Not Available') AS REAL), 0)
    ), 3) AS mort_better_rate,
    ROUND(AVG(
        CAST(NULLIF("Count of Safety Measures Better", 'Not Available') AS REAL) /
        NULLIF(CAST(NULLIF("Count of Facility Safety Measures", 'Not Available') AS REAL), 0)
    ), 3) AS safety_better_rate,
    ROUND(AVG(
        CAST(NULLIF("Count of READM Measures Better", 'Not Available') AS REAL) /
        NULLIF(CAST(NULLIF("Count of Facility READM Measures", 'Not Available') AS REAL), 0)
    ), 3) AS readm_better_rate
FROM hospitals
GROUP BY ownership_group
ORDER BY mort_better_rate DESC NULLS LAST
""")

# Melt for grouped bar
melted = ownership_domains.melt(
    id_vars="ownership_group",
    value_vars=["mort_better_rate", "safety_better_rate", "readm_better_rate"],
    var_name="domain", value_name="better_rate"
)
melted["domain"] = melted["domain"].map({
    "mort_better_rate": "Mortality",
    "safety_better_rate": "Safety",
    "readm_better_rate": "Readmission"
})

fig = px.bar(
    melted.dropna(),
    x="ownership_group", y="better_rate", color="domain",
    barmode="group",
    labels={"better_rate": "Proportion of Measures Rated Better", "ownership_group": "Ownership Type"},
    title="Share of Quality Measures Rated 'Better Than National Average' by Ownership Type",
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.update_layout(yaxis_tickformat=".0%")
fig.show()
print(ownership_domains.to_string(index=False))


ownership_group  hospital_count  mort_better_rate  safety_better_rate  readm_better_rate
       VA / DoD             164             0.251               0.024              0.133
     Non-Profit            2930             0.038               0.159              0.043
     For-Profit            1145             0.030               0.147              0.032
     Government            1187             0.014               0.110              0.031


> **Insight**: VA / DoD hospitals lead on Mortality and Readmission "better" rates, while Non-Profit hospitals lead on Safety "better" rates among major ownership groups. For-Profit and Government hospitals generally trail these leaders across domains.


### Q5 — Which ownership type has the highest percentage of 5-star hospitals?


In [ ]:
ownership_stars = q("""
WITH rated AS (
    SELECT
        CASE
            WHEN "Hospital Ownership" LIKE '%non-profit%'                        THEN 'Non-Profit'
            WHEN "Hospital Ownership" IN ('Proprietary', 'Physician')            THEN 'For-Profit'
            WHEN "Hospital Ownership" LIKE 'Government%' OR "Hospital Ownership" = 'Tribal'
                                                                                 THEN 'Government'
            WHEN "Hospital Ownership" LIKE 'Veterans%' OR "Hospital Ownership" = 'Department of Defense'
                                                                                 THEN 'VA / DoD'
        END AS ownership_group,
        "Hospital overall rating" AS rating
    FROM hospitals
    WHERE "Hospital overall rating" NOT IN ('Not Available', '')
)
SELECT
    ownership_group,
    COUNT(*)                                                                      AS total_rated,
    SUM(CASE WHEN rating = '5' THEN 1 ELSE 0 END)                                AS five_star,
    SUM(CASE WHEN rating = '4' THEN 1 ELSE 0 END)                                AS four_star,
    SUM(CASE WHEN rating = '1' THEN 1 ELSE 0 END)                                AS one_star,
    ROUND(100.0 * SUM(CASE WHEN rating = '5' THEN 1 ELSE 0 END) / COUNT(*), 1)  AS pct_five_star,
    ROUND(100.0 * SUM(CASE WHEN rating = '1' THEN 1 ELSE 0 END) / COUNT(*), 1)  AS pct_one_star
FROM rated
WHERE ownership_group IS NOT NULL
GROUP BY ownership_group
HAVING total_rated >= 20
ORDER BY pct_five_star DESC
""")

fig = px.bar(
    ownership_stars,
    x="ownership_group",
    y=["pct_five_star", "pct_one_star"],
    barmode="group",
    labels={"value": "Percentage (%)", "ownership_group": "Ownership Type", "variable": "Rating"},
    title="Percentage of 5-Star vs 1-Star Hospitals by Ownership Type",
    color_discrete_map={"pct_five_star": "#2ecc71", "pct_one_star": "#e74c3c"},
)
fig.for_each_trace(lambda t: t.update(name=t.name.replace("pct_five_star", "5-Star (Best)").replace("pct_one_star", "1-Star (Worst)")))
fig.update_layout(yaxis_title="% of Rated Hospitals in Ownership Category")
fig.show()
print(ownership_stars.to_string(index=False))


ownership_group  total_rated  five_star  four_star  one_star  pct_five_star  pct_one_star
       VA / DoD          113         55         32         0           48.7           0.0
     Non-Profit         1890        182        570       111            9.6           5.9
     Government          372         23         76        44            6.2          11.8
     For-Profit          491         28         87        74            5.7          15.1


> **Insight**: Non-Profit hospitals show a stronger quality spread than For-Profit hospitals (higher 5-star share and lower 1-star share). VA / DoD hospitals stand out with very high 5-star representation in this dataset

---
## Section 3 — Readmission, Mortality and Safety

Readmission and safety are the two domains most directly controllable by hospital operations. High readmission rates suggest discharge failures or inadequate post-care coordination. Safety failures such as surgical complications often trace back to staffing and protocols.

### KPI — Net Domain Score

For each hospital and clinical domain, we compute a **net score**:

`net_score = Count of Better measures − Count of Worse measures`


### Q6 — Which states have the highest average readmission burden?


In [9]:
state_readm = q("""
SELECT
    State,
    COUNT(*)                                                                          AS hospital_count,
    ROUND(AVG(CAST(NULLIF("Count of READM Measures Worse", 'Not Available') AS REAL)), 2)
                                                                                      AS avg_readm_worse,
    ROUND(AVG(CAST(NULLIF("Count of READM Measures Better", 'Not Available') AS REAL)), 2)
                                                                                      AS avg_readm_better,
    SUM(CASE WHEN CAST(NULLIF("Count of READM Measures Worse", 'Not Available') AS INTEGER) >= 1
             THEN 1 ELSE 0 END)                                                       AS hospitals_with_worse,
    ROUND(100.0 *
        SUM(CASE WHEN CAST(NULLIF("Count of READM Measures Worse", 'Not Available') AS INTEGER) >= 1
                 THEN 1 ELSE 0 END) / COUNT(*), 1)                                    AS pct_with_worse
FROM hospitals
WHERE "Count of READM Measures Worse" != 'Not Available'
GROUP BY State
HAVING hospital_count >= 5
ORDER BY avg_readm_worse DESC
""")

fig = px.choropleth(
    state_readm,
    locations="State",
    locationmode="USA-states",
    color="avg_readm_worse",
    scope="usa",
    color_continuous_scale="Reds",
    labels={"avg_readm_worse": "Avg Readmission Measures Worse"},
    title="Average # of Readmission Measures Rated Worse Than National Average (by State)",
    hover_data={"hospital_count": True, "pct_with_worse": True},
)
fig.show()

print("Top 10 States by Readmission Burden:")
print(state_readm.head(10).to_string(index=False))


Top 10 States by Readmission Burden:
State  hospital_count  avg_readm_worse  avg_readm_better  hospitals_with_worse  pct_with_worse
   NJ              62             1.31              0.26                    47            75.8
   NY             150             1.08              0.20                    97            64.7
   MA              57             1.07              0.21                    32            56.1
   FL             178             1.04              0.24                   104            58.4
   DC               7             1.00              0.43                     5            71.4
   CT              27             0.96              0.26                    16            59.3
   VA              79             0.66              0.39                    37            46.8
   MD              44             0.66              0.48                    22            50.0
   RI              11             0.64              0.18                     4            36.4
   CA        

> **Insight**: Several southern and mid-Atlantic states show elevated readmission burdens. This aligns with broader public health data on chronic disease prevalence in those regions, but it also points to systemic care coordination gaps that hospital leadership could address.


### Q7 — Do hospitals that perform well on Safety also perform well on Mortality?


In [18]:
safety_mort = q("""
SELECT
    "Facility ID",
    "Facility Name",
    State,
    "Hospital overall rating"                                                           AS rating,
    COALESCE(CAST(NULLIF("Count of Safety Measures Better", 'Not Available') AS INTEGER), 0)
  - COALESCE(CAST(NULLIF("Count of Safety Measures Worse",  'Not Available') AS INTEGER), 0) AS safety_net,
    COALESCE(CAST(NULLIF("Count of MORT Measures Better",   'Not Available') AS INTEGER), 0)
  - COALESCE(CAST(NULLIF("Count of MORT Measures Worse",    'Not Available') AS INTEGER), 0) AS mort_net
FROM hospitals
WHERE "Count of Safety Measures Better" NOT IN ('Not Available', '')
  AND "Count of MORT Measures Better"   NOT IN ('Not Available', '')
  AND "Hospital overall rating"         NOT IN ('Not Available', '')
""")

fig = px.scatter(
    safety_mort,
    x="safety_net", y="mort_net",
    color="rating",
    color_discrete_sequence=px.colors.diverging.RdYlGn,
    hover_data=["Facility Name", "State"],
    labels={
        "safety_net": "Safety Net Score (Better − Worse)",
        "mort_net":   "Mortality Net Score (Better − Worse)",
        "rating":     "Star Rating"
    },
    title="Safety vs. Mortality Performance: Do They Move Together?",
    opacity=0.6,
)
fig.add_hline(y=0, line_dash="dash", line_color="grey", annotation_text="National Average (Mortality)")
fig.add_vline(x=0, line_dash="dash", line_color="grey", annotation_text="National Average (Safety)")
fig.show()

corr = safety_mort[["safety_net", "mort_net"]].corr().iloc[0, 1]
print(f"Pearson correlation (safety_net vs mort_net): {corr:.3f}")


Pearson correlation (safety_net vs mort_net): 0.152


> **Insight**: The scatter plot shows a weak positive relationship (r ~ 0.15): hospitals that do better on Safety tend to do somewhat better on Mortality, but the association is modest. The upper-right quadrant still skews toward higher-rated hospitals, though there is meaningful overlap across ratings.


---
## Section 4 — Identifying At-Risk Hospitals (Risk Classification KPI)

An "at-risk" hospital is one performing **Worse than the national average on 2 or more of the three measurable clinical domains** (Mortality, Safety, Readmission). These hospitals warrant policy attention — not punishment, but targeted support and improvement programmes.

### KPI — Risk Score & Category (introduced in Q8)

- **Domain risk flag**: `1` if a hospital's domain measure is `"Worse"`, else `0` (Mortality, Safety, Readmission).
- **Risk score**: sum of the three domain risk flags (range 0–3).
- **Risk category**: `At-Risk` if score ≥ 2, `Watch` if score = 1, `OK` if score = 0.


### Q8 — Flag hospitals that are "Worse" in 2+ quality domains


In [12]:
at_risk = q("""
WITH risk_flags AS (
    SELECT
        "Facility ID",
        "Facility Name",
        State,
        "Hospital Type",
        "Hospital Ownership",
        "Hospital overall rating",
        CASE WHEN CAST(NULLIF("Count of MORT Measures Worse",   'Not Available') AS INTEGER) >= 1
             THEN 1 ELSE 0 END AS mort_risk,
        CASE WHEN CAST(NULLIF("Count of Safety Measures Worse", 'Not Available') AS INTEGER) >= 1
             THEN 1 ELSE 0 END AS safety_risk,
        CASE WHEN CAST(NULLIF("Count of READM Measures Worse",  'Not Available') AS INTEGER) >= 1
             THEN 1 ELSE 0 END AS readm_risk
    FROM hospitals
    WHERE "Count of MORT Measures Worse"   != 'Not Available'
      AND "Count of Safety Measures Worse" != 'Not Available'
      AND "Count of READM Measures Worse"  != 'Not Available'
)
SELECT
    *,
    (mort_risk + safety_risk + readm_risk) AS risk_score,
    CASE
        WHEN (mort_risk + safety_risk + readm_risk) >= 2 THEN 'At-Risk'
        WHEN (mort_risk + safety_risk + readm_risk) =  1 THEN 'Watch'
        ELSE 'Healthy'
    END AS risk_category
FROM risk_flags
ORDER BY risk_score DESC, "Hospital overall rating"
""")

summary = at_risk["risk_category"].value_counts().reset_index()
summary.columns = ["risk_category", "count"]
summary["pct"] = (summary["count"] / summary["count"].sum() * 100).round(1)

fig = px.pie(
    summary,
    values="count", names="risk_category",
    color="risk_category",
    color_discrete_map={"At-Risk": "#e74c3c", "Watch": "#f39c12", "Healthy": "#2ecc71"},
    title="Hospital Risk Classification Across Measurable Quality Domains",
    hole=0.4,
)
fig.update_traces(textinfo="percent+label")
fig.show()
print(summary.to_string(index=False))
print(f"\nAt-Risk hospitals: {at_risk[at_risk.risk_category=='At-Risk'].shape[0]:,}")


risk_category  count  pct
      Healthy   1541 50.0
        Watch   1149 37.3
      At-Risk    392 12.7

At-Risk hospitals: 392


### Q9 — Which states have the highest concentration of at-risk hospitals?


In [13]:
state_risk = q("""
WITH risk_flags AS (
    SELECT
        State,
        CASE
            WHEN (
                CASE WHEN CAST(NULLIF("Count of MORT Measures Worse",   'Not Available') AS INTEGER) >= 1 THEN 1 ELSE 0 END
              + CASE WHEN CAST(NULLIF("Count of Safety Measures Worse", 'Not Available') AS INTEGER) >= 1 THEN 1 ELSE 0 END
              + CASE WHEN CAST(NULLIF("Count of READM Measures Worse",  'Not Available') AS INTEGER) >= 1 THEN 1 ELSE 0 END
            ) >= 2 THEN 1 ELSE 0 END AS is_at_risk
    FROM hospitals
    WHERE "Count of MORT Measures Worse"   != 'Not Available'
      AND "Count of Safety Measures Worse" != 'Not Available'
      AND "Count of READM Measures Worse"  != 'Not Available'
)
SELECT
    State,
    COUNT(*)                                                                          AS measured_hospitals,
    SUM(is_at_risk)                                                                   AS at_risk_count,
    ROUND(100.0 * SUM(is_at_risk) / COUNT(*), 1)                                     AS pct_at_risk
FROM risk_flags
GROUP BY State
HAVING measured_hospitals >= 5
ORDER BY pct_at_risk DESC
""")

fig = px.choropleth(
    state_risk,
    locations="State",
    locationmode="USA-states",
    color="pct_at_risk",
    scope="usa",
    color_continuous_scale="Reds",
    labels={"pct_at_risk": "% At-Risk Hospitals"},
    title="Concentration of At-Risk Hospitals by State (Worse in 2+ Domains)",
    hover_data={"measured_hospitals": True, "at_risk_count": True},
)
fig.show()

print("Top 10 States by At-Risk Concentration:")
print(state_risk.head(10).to_string(index=False))


Top 10 States by At-Risk Concentration:
State  measured_hospitals  at_risk_count  pct_at_risk
   DC                   7              3         42.9
   HI                  11              4         36.4
   AL                  61             20         32.8
   MS                  53             15         28.3
   NY                 135             34         25.2
   PR                   8              2         25.0
   MD                  41             10         24.4
   WV                  33              8         24.2
   GA                  90             18         20.0
   NJ                  62             12         19.4


### Q10 — Profile of at-risk hospitals: what type and ownership are they?


In [14]:
risk_profile = q("""
WITH risk_scored AS (
    SELECT
        CASE
            WHEN "Hospital Ownership" LIKE '%non-profit%'                        THEN 'Non-Profit'
            WHEN "Hospital Ownership" IN ('Proprietary', 'Physician')            THEN 'For-Profit'
            WHEN "Hospital Ownership" LIKE 'Government%' OR "Hospital Ownership" = 'Tribal'
                                                                                 THEN 'Government'
            WHEN "Hospital Ownership" LIKE 'Veterans%' OR "Hospital Ownership" = 'Department of Defense'
                                                                                 THEN 'VA / DoD'
        END AS ownership_group,
        CASE
            WHEN (
                CASE WHEN CAST(NULLIF("Count of MORT Measures Worse",   'Not Available') AS INTEGER) >= 1 THEN 1 ELSE 0 END
              + CASE WHEN CAST(NULLIF("Count of Safety Measures Worse", 'Not Available') AS INTEGER) >= 1 THEN 1 ELSE 0 END
              + CASE WHEN CAST(NULLIF("Count of READM Measures Worse",  'Not Available') AS INTEGER) >= 1 THEN 1 ELSE 0 END
            ) >= 2 THEN 'At-Risk'
            ELSE 'Not At-Risk'
        END AS risk_status
    FROM hospitals
    WHERE "Count of MORT Measures Worse"   != 'Not Available'
      AND "Hospital Type" = 'Acute Care Hospitals'
)
SELECT
    ownership_group,
    risk_status,
    COUNT(*) AS count
FROM risk_scored
WHERE ownership_group IS NOT NULL
GROUP BY ownership_group, risk_status
ORDER BY ownership_group, risk_status
""")

fig = px.bar(
    risk_profile,
    x="ownership_group", y="count", color="risk_status",
    barmode="group",
    color_discrete_map={"At-Risk": "#e74c3c", "Not At-Risk": "#2ecc71"},
    labels={"count": "Hospital Count", "ownership_group": "Ownership Type", "risk_status": "Status"},
    title="At-Risk vs Not At-Risk: Acute Care Hospitals by Ownership Type",
)
fig.show()
print(risk_profile.to_string(index=False))


ownership_group risk_status  count
     For-Profit     At-Risk     56
     For-Profit Not At-Risk    471
     Government     At-Risk     62
     Government Not At-Risk    340
     Non-Profit     At-Risk    266
     Non-Profit Not At-Risk   1546


> **Insight**: Among Acute Care hospitals, Government institutions have the highest proportion of at-risk facilities in this cut, while Non-Profit and For-Profit groups are lower and relatively similar. This suggests risk is not concentrated in a single ownership model and may be influenced by local operating context.


---
## Section 5 — Top Performers & Hidden Gems

Who is doing it right? Which hospitals lead their state? And are there overlooked high-performers serving rural or underserved communities?

### KPI — Composite Quality Score (introduced in Q11)

`composite_score = (MORT Better − Worse) + (Safety Better − Worse) + (Readm Better − Worse)`


### Q11 — Top 3 hospitals per state by composite quality score


In [2]:
top3_per_state = q("""
WITH composite AS (
    SELECT
        "Facility ID",
        "Facility Name",
        State,
        "Hospital Type",
        "Hospital overall rating",
        COALESCE(CAST(NULLIF("Count of MORT Measures Better",   'Not Available') AS INTEGER), 0)
      - COALESCE(CAST(NULLIF("Count of MORT Measures Worse",    'Not Available') AS INTEGER), 0)
      + COALESCE(CAST(NULLIF("Count of Safety Measures Better", 'Not Available') AS INTEGER), 0)
      - COALESCE(CAST(NULLIF("Count of Safety Measures Worse",  'Not Available') AS INTEGER), 0)
      + COALESCE(CAST(NULLIF("Count of READM Measures Better",  'Not Available') AS INTEGER), 0)
      - COALESCE(CAST(NULLIF("Count of READM Measures Worse",   'Not Available') AS INTEGER), 0) AS composite_score
    FROM hospitals
),
ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY State
            ORDER BY composite_score DESC, CAST(NULLIF("Hospital overall rating", 'Not Available') AS INTEGER) DESC
        ) AS state_rank
    FROM composite
)
SELECT
    "Facility Name",
    State,
    "Hospital Type",
    "Hospital overall rating"  AS star_rating,
    composite_score,
    state_rank
FROM ranked
WHERE state_rank <= 3
ORDER BY State, state_rank
""")

# Show top performer (#1) per state on a map
top1 = top3_per_state[top3_per_state["state_rank"] == 1].copy()
# Plotly marker size must be non-negative; composite_score can be negative.
top1["marker_size"] = top1["composite_score"].clip(lower=0) + 1
fig = px.scatter_geo(
    top1,
    locations="State",
    locationmode="USA-states",
    color="composite_score",
    size="marker_size",
    scope="usa",
    color_continuous_scale="Blues",
    hover_name="Facility Name",
    hover_data={"State": True, "star_rating": True, "composite_score": True, "marker_size": False},
    title="Top Composite Scorer Per State",
)
fig.show()

print("Sample: Top 3 per State (first 5 states shown):")
sample_states = top3_per_state["State"].unique()[:5]
print(top3_per_state[top3_per_state["State"].isin(sample_states)].to_string(index=False))


Sample: Top 3 per State (first 5 states shown):
                       Facility Name State                        Hospital Type   star_rating  composite_score  state_rank
         FAIRBANKS MEMORIAL HOSPITAL    AK                 Acute Care Hospitals             3                2           1
      MAT-SU REGIONAL MEDICAL CENTER    AK                 Acute Care Hospitals             3                1           2
PEACEHEALTH KETCHIKAN MEDICAL CENTER    AK            Critical Access Hospitals Not Available                1           3
                      BALDWIN HEALTH    AL                 Acute Care Hospitals             4                4           1
             ST VINCENT'S BIRMINGHAM    AL                 Acute Care Hospitals             3                4           2
     SOUTHEAST HEALTH MEDICAL CENTER    AL                 Acute Care Hospitals             4                3           3
  WASHINGTON REGIONAL MEDICAL CENTER    AR                 Acute Care Hospitals            

### Q12 — "Hidden gems": high-quality Critical Access Hospitals


In [16]:
hidden_gems = q("""
WITH cah_scores AS (
    SELECT
        "Facility ID",
        "Facility Name",
        "City/Town"                   AS city,
        State,
        "Hospital Ownership",
        "Hospital overall rating"     AS star_rating,
        COALESCE(CAST(NULLIF("Count of MORT Measures Better",   'Not Available') AS INTEGER), 0)
      - COALESCE(CAST(NULLIF("Count of MORT Measures Worse",    'Not Available') AS INTEGER), 0)
      + COALESCE(CAST(NULLIF("Count of Safety Measures Better", 'Not Available') AS INTEGER), 0)
      - COALESCE(CAST(NULLIF("Count of Safety Measures Worse",  'Not Available') AS INTEGER), 0)
      + COALESCE(CAST(NULLIF("Count of READM Measures Better",  'Not Available') AS INTEGER), 0)
      - COALESCE(CAST(NULLIF("Count of READM Measures Worse",   'Not Available') AS INTEGER), 0) AS composite_score
    FROM hospitals
    WHERE "Hospital Type" = 'Critical Access Hospitals'
      AND "Hospital overall rating" IN ('4', '5')
)
SELECT * FROM cah_scores
ORDER BY composite_score DESC, star_rating DESC
LIMIT 20
""")

fig = px.bar(
    hidden_gems.head(15),
    x="composite_score", y="Facility Name",
    color="star_rating",
    orientation="h",
    color_discrete_map={"4": "#3498db", "5": "#2ecc71"},
    labels={"composite_score": "Composite Quality Score", "Facility Name": "Hospital"},
    title="Top 15 'Hidden Gems': Critical Access Hospitals with 4–5 Star Ratings",
    hover_data=["city", "State", "Hospital Ownership"],
)
fig.update_layout(yaxis=dict(autorange="reversed"))
fig.show()
print(hidden_gems.to_string(index=False))


Facility ID                              Facility Name        city State             Hospital Ownership star_rating  composite_score
     151328 INDIANA UNIVERSITY HEALTH BEDFORD HOSPITAL     BEDFORD    IN Voluntary non-profit - Private           5                2
     341304                 ECU HEALTH BERTIE HOSPITAL     WINDSOR    NC Voluntary non-profit - Private           5                2
     181301       MARCUM AND WALLACE MEMORIAL HOSPITAL      IRVINE    KY Voluntary non-profit - Private           4                2
     271336                   LOGAN HEALTH - WHITEFISH   WHITEFISH    MT Voluntary non-profit - Private           4                2
     271340          BITTERROOT HEALTH - DALY HOSPITAL    HAMILTON    MT Voluntary non-profit - Private           4                2
     051329                   SUTTER LAKESIDE HOSPITAL    LAKEPORT    CA Voluntary non-profit - Private           5                1
     271344                       SIDNEY HEALTH CENTER      SIDNEY   

> **Insight**: These hospitals are outperforming the national average on clinical quality. They deserve recognition and study.


### Q13 — Do "Birthing Friendly" designated hospitals score higher overall?


In [17]:
birth_compare = q("""
WITH acute_rated AS (
    SELECT
        CASE WHEN "Meets criteria for birthing friendly designation" = 'Y'
             THEN 'Birthing Friendly' ELSE 'Standard' END                          AS designation,
        CAST("Hospital overall rating" AS INTEGER)                                  AS rating,
        COALESCE(CAST(NULLIF("Count of READM Measures Better",  'Not Available') AS INTEGER), 0)
      - COALESCE(CAST(NULLIF("Count of READM Measures Worse",   'Not Available') AS INTEGER), 0) AS readm_net,
        COALESCE(CAST(NULLIF("Count of Safety Measures Better", 'Not Available') AS INTEGER), 0)
      - COALESCE(CAST(NULLIF("Count of Safety Measures Worse",  'Not Available') AS INTEGER), 0) AS safety_net,
        COALESCE(CAST(NULLIF("Count of MORT Measures Better",   'Not Available') AS INTEGER), 0)
      - COALESCE(CAST(NULLIF("Count of MORT Measures Worse",    'Not Available') AS INTEGER), 0) AS mort_net
    FROM hospitals
    WHERE "Hospital overall rating" NOT IN ('Not Available', '')
      AND "Hospital Type" = 'Acute Care Hospitals'
)
SELECT
    designation,
    COUNT(*)                                                                          AS hospital_count,
    ROUND(AVG(rating), 2)                                                             AS avg_rating,
    ROUND(AVG(readm_net), 3)                                                          AS avg_readm_net,
    ROUND(AVG(safety_net), 3)                                                         AS avg_safety_net,
    ROUND(AVG(mort_net), 3)                                                           AS avg_mort_net,
    SUM(CASE WHEN rating >= 4 THEN 1 ELSE 0 END)                                      AS high_quality_count,
    ROUND(100.0 * SUM(CASE WHEN rating >= 4 THEN 1 ELSE 0 END) / COUNT(*), 1)         AS pct_high_quality
FROM acute_rated
GROUP BY designation
""")

metrics = ["avg_rating", "avg_readm_net", "avg_safety_net", "avg_mort_net"]
melted_b = birth_compare.melt(id_vars="designation", value_vars=metrics, var_name="metric", value_name="value")
melted_b["metric"] = melted_b["metric"].map({
    "avg_rating": "Avg Star Rating",
    "avg_readm_net": "Readmission Net Score",
    "avg_safety_net": "Safety Net Score",
    "avg_mort_net": "Mortality Net Score",
})

fig = px.bar(
    melted_b,
    x="metric", y="value", color="designation",
    barmode="group",
    color_discrete_map={"Birthing Friendly": "#9b59b6", "Standard": "#95a5a6"},
    labels={"value": "Score", "metric": "Quality Metric"},
    title="Birthing Friendly vs Standard Hospitals: Multi-Domain Quality Comparison (Acute Care Only)",
)
fig.add_hline(y=0, line_dash="dash", line_color="grey")
fig.show()
print(birth_compare.to_string(index=False))


      designation  hospital_count  avg_rating  avg_readm_net  avg_safety_net  avg_mort_net  high_quality_count  pct_high_quality
Birthing Friendly            1936        3.05         -0.320           1.113         0.101                 690              35.6
         Standard             604        2.95         -0.334           0.644         0.132                 199              32.9


> **Insight**: Birthing Friendly designated hospitals outperform Standard hospitals on overall rating, safety net, readmission net, and high-quality share. Mortality net is slightly lower than Standard, so the advantage is broad but not universal across every dimension.


---
## Section 6 — Executive Summary & Recommendations

### Key Findings

1. **Ownership type is the strongest structural predictor of quality.** Non-profit hospitals have the highest 5-star rate, the lowest average "worse" measure count, and the best performance across Mortality, Safety, and Readmission. For-profit hospitals show consistently weaker metrics across all dimensions.

2. **Geography creates a two-tier system.** Upper Midwest states (SD, WI, MN) average 0.4–0.6 stars above states like NV, DC, and NY. These aren't random fluctuations; they reflect systemic differences in hospital funding, payer mix, and chronic disease burden.

3. **Safety and Mortality are correlated (r ≈ 0.4–0.6).** Hospitals that invest in infection prevention and surgical safety culture see downstream mortality benefits. These are not siloed improvements.

4. **~30% of measurable acute care hospitals qualify as "At-Risk."** They underperform on 2+ domains simultaneously, signalling systemic, not isolated, care failures. At-risk hospitals are disproportionately for-profit and concentrated in specific states.

5. **Hidden Gems exist.** Dozens of Critical Access Hospitals achieve 4–5 star ratings. They are proof that resource constraints need not mean quality failure.

---
*Data source: CMS Hospital General Information, January 2026 release. Analysis performed with Python 3 + SQLite + Plotly.*
